In [2]:
# --- Fix Render Port Scanning 405 Errors ---
import tornado.web
tornado.web.RequestHandler.head = tornado.web.RequestHandler.get
import io, base64, difflib, os
import pandas as pd
import ipywidgets as widgets
import pycountry
from IPython.display import display, HTML, clear_output

# --- Fallback Global Variables and Functions ---
SOCCERDATA_POSITIONS = {}  # Add name-to-position mappings here if needed

def lookup_country(abbr):
    """Safely converts country codes (alpha-2 or alpha-3) to full names."""
    abbr = str(abbr).strip().upper()
    if not abbr or abbr in ['NAN', 'NONE', '0']:
        return "", ""
    try:
        country = pycountry.countries.get(alpha_3=abbr) or pycountry.countries.get(alpha_2=abbr)
        if country:
            return country.name, country.alpha_3
    except Exception:
        pass
    return abbr, abbr

PLAYER_COLS = [1, 4, 7]
TOTAL_TABLES = 28

def lookup_player(name, t_num):
    if 18 <= t_num <= 22: return name, "Defender"
    if 23 <= t_num <= 28: return name, "Goalkeeper"
    if name in SOCCERDATA_POSITIONS: return name, SOCCERDATA_POSITIONS[name]
    matches = difflib.get_close_matches(name, SOCCERDATA_POSITIONS.keys(), n=1, cutoff=0.6)
    if matches: return matches[0], SOCCERDATA_POSITIONS[matches[0]]
    return name, "Midfielder/Forward"

print("✅ Engine and configurations successfully loaded.")
menu_output = widgets.Output()
display(menu_output)

def auto_load_file(filename="World Cup PlayBook26.xlsx"):
    with menu_output:
        clear_output()
        if not os.path.exists(filename):
            print(f"❌ Error: The file '{filename}' was not found in the current directory.")
            return
            
        try:
            print(f"🔄 Automatically loading file: {filename}")
            xl = pd.ExcelFile(filename)
            sheet = next((s for s in xl.sheet_names if "summary" in s.lower().strip()), xl.sheet_names[0])
            df = pd.read_excel(filename, sheet_name=sheet, header=None)
            
            tables_dict = {}
            num_rows, num_cols = df.shape
            
            for p_col in PLAYER_COLS:
                if p_col >= num_cols: continue
                for r_idx in range(num_rows):
                    idx_cell = str(df.iloc[r_idx, p_col - 1]).strip().replace('.0', '')
                    if idx_cell.isdigit():
                        t_num, players = int(idx_cell), []
                        for off in range(6):
                            if r_idx + off < num_rows:
                                p_name = str(df.iloc[r_idx + off, p_col]).strip()
                                c_abbr = str(df.iloc[r_idx + off, p_col + 1]).strip().upper() if p_col + 1 < num_cols else ""
                                if p_name and p_name.lower() not in ['nan', 'none', '0', '0.0'] and not p_name.replace('.', '').isdigit():
                                    c_name, clean_pos = lookup_player(p_name, t_num)
                                    c_fullname, final_abbr = lookup_country(c_abbr)
                                    players.append({
                                        "display_text": f"{c_name} ({final_abbr})" if final_abbr and final_abbr != "NAN" else c_name,
                                        "clean_name": c_name, "position": clean_pos, "country": c_fullname
                                    })
                        if players: tables_dict[t_num] = players

            discovered_tables = sorted(list(tables_dict.keys()))
            if not discovered_tables:
                print("❌ No valid player groups detected."); return

            name_input = widgets.Text(description="✍️ Your Name:", placeholder="Full Name...", layout=widgets.Layout(width='45%'))
            email_input = widgets.Text(description="📧 Your Email:", placeholder="Email...", layout=widgets.Layout(width='45%'))
            display(widgets.HBox([name_input, email_input], layout=widgets.Layout(margin='10px 0px 20px 0px')))
            
            widget_items = []
            for t_num in discovered_tables:
                p_objs = tables_dict[t_num]
                dropdown = widgets.Dropdown(options=[(p["display_text"], p) for p in p_objs], description=f"Group {t_num}: ", style={'description_width': '80px'}, layout=widgets.Layout(width='95%'))
                widget_items.append(dropdown)

            winner_box = widgets.Text(description="World Cup Winner:", placeholder="Country Name...", style={'description_width': '130px'}, layout=widgets.Layout(width='90%'))
            finalist_box = widgets.Text(description="World Cup Finalist:", placeholder="Country Name...", style={'description_width': '130px'}, layout=widgets.Layout(width='90%'))
            semi_box = widgets.Text(description="Semi Finalist:", placeholder="Country Name...", style={'description_width': '130px'}, layout=widgets.Layout(width='90%'))

            split_mark = max(1, len(discovered_tables) // 2)
            display(widgets.HBox([widgets.VBox(widget_items[:split_mark]), widgets.VBox(widget_items[split_mark:] + [widgets.HTML("<br><b>🏅 Bracket Predictions:</b>"), winner_box, finalist_box, semi_box])]))

            action_button = widgets.Button(description="📋 Compile All Selections", button_style='success', layout=widgets.Layout(width='340px', margin='25px 0px 0px 0px'))
            results_view = widgets.Output()
            display(action_button, results_view)

            def generate_excel_output(btn):
                with results_view:
                    clear_output()
                    v_name, v_email, v_win, v_fin, v_semi = name_input.value.strip(), email_input.value.strip(), winner_box.value.strip(), finalist_box.value.strip(), semi_box.value.strip()
                    
                    missing = [label for label, val in [("Your Name", v_name), ("Your Email", v_email), ("Winner", v_win), ("Finalist", v_fin), ("Semi Finalist", v_semi)] if not val]
                    if missing:
                        display(HTML(f"<b style='color:red;'>❌ ERROR: Missing Required Information: {', '.join(missing)}</b>")); return
                    
                    # Core Data Construction
                    final_rows = [
                        {"Selection Objective": "Participant Name", "Your Pick": v_name, "Player Position": "", "Country": ""},
                        {"Selection Objective": "Participant Email", "Your Pick": v_email, "Player Position": "", "Country": ""}
                    ]
                    for d in widget_items:
                        s_p = d.value
                        final_rows.append({"Selection Objective": d.description.replace(':', '').strip(), "Your Pick": s_p["clean_name"], "Player Position": s_p["position"], "Country": s_p["country"]})
                    
                    for label, val in [("World Cup Winner", v_win), ("World Cup Finalist", v_fin), ("Semi Finalist", v_semi)]:
                        final_rows.append({"Selection Objective": label, "Your Pick": val, "Player Position": "", "Country": ""})
                    
                    out_df = pd.DataFrame(final_rows)
                    
                    # 1. Create Excel Download File with Header Row added at the top
                    out_stream = io.BytesIO()
                    with pd.ExcelWriter(out_stream, engine='openpyxl') as writer:
                        # Write Custom Title string at cell A1
                        pd.DataFrame([["Moe's Tavern World Cup Pool"]]).to_excel(writer, index=False, header=False, sheet_name='Selections', startrow=0, startcol=0)
                        # Append the main structured table 2 rows below the header title
                        out_df.to_excel(writer, index=False, sheet_name='Selections', startrow=2)
                    out_stream.seek(0)
                    b64 = base64.b64encode(out_stream.read()).decode()
                    
                    # 2. Build Copy-Pasteable HTML View including the custom header row element
                    html_table = out_df.to_html(index=False, classes="preview-table")
                    styled_html = f"""
                    <style>
                        .pool-banner {{ font-family: sans-serif; font-size: 1.6em; font-weight: bold; color: #2c3e50; margin: 15px 0px 5px 0px; }}
                        .preview-table {{ border-collapse: collapse; width: 100%; font-family: sans-serif; }}
                        .preview-table th, .phreview-table td {{ border: 1px solid #dddddd; text-align: left; padding: 8px; }}
                        .preview-table th {{ background-color: #f2f2f2; font-weight: bold; }}
                        .preview-table tr:nth-child(even) {{ background-color: #fafafa; }}
                    </style>
                    <br><b style='color:green; font-size:1.1em;'>✅ Compiled selections for {v_name}!</b><br>
                    <p style="color:#555; font-size:0.95em;">💡 You can select, copy, and paste the table below directly into Excel, or download the formal file link.</p>
                    <a href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}" download="Selections_{v_name.replace(" ", "_")}.xlsx" style="font-weight:bold; color:#1e7e34; text-decoration:underline; font-size:1.05em;">💾 Click here to download your Excel file</a>
                    <br><br>
                    <div class="pool-banner">Moe's Tavern World Cup Pool</div>
                    {html_table}
                    """
                    display(HTML(styled_html))
                    
            action_button.on_click(generate_excel_output)
        except Exception as err:
            print(f"❌ Error processing Excel: {err}")

# Trigger the process automatically on script load
auto_load_file()

✅ Engine and configurations successfully loaded.


Output()